In [ ]:
# --------------------------
# CELL 1 — imports & config
# --------------------------
from __future__ import annotations

import os, sys, csv, glob, pathlib
from pathlib import Path
from typing import List, Set, Optional
import pandas as pd
import duckdb

_ANALYTICS_SHARED = Path.cwd().resolve().parents[2] / "Analytics" / "shared"
if str(_ANALYTICS_SHARED) not in sys.path:
    sys.path.insert(0, str(_ANALYTICS_SHARED))

from locations import Location

DUCK_THREADS = min(8, os.cpu_count() or 1)
DUCK_MEM = "8GB"
DUCKDB_PATH = "new_wallet_first_seen_appendonly.duckdb"

def get_duck(db_path: str = DUCKDB_PATH, threads: int = DUCK_THREADS, mem: str = DUCK_MEM) -> duckdb.DuckDBPyConnection:
    con = duckdb.connect(db_path)
    con.execute(f"PRAGMA threads={threads};")
    con.execute(f"PRAGMA memory_limit='{mem}';")
    con.execute("PRAGMA enable_progress_bar=false;")
    return con


# ---------------------------------------
# CELL 2 — whitelist helper (same as yours)
# ---------------------------------------
def load_token_whitelist_from_status_csv(csv_path: str) -> Set[str]:
    if not os.path.exists(csv_path):
        print(f"Token status CSV not found: {csv_path}")
        return set()

    df = pd.read_csv(csv_path, dtype=str)

    def norm(s): return str(s).strip().lower()

    mask = (df.get("is_erc_20", "").map(norm) == "true") & \
           (df.get("local_status", "").map(norm) == "pass") & \
           (df.get("price_data_status", "").map(norm) == "success")

    if "contract_address" not in df.columns:
        print("'contract_address' column missing in token status CSV")
        return set()

    addrs = (df.loc[mask, "contract_address"]
               .dropna()
               .astype(str)
               .str.strip()
               .str.lower()
               .tolist())
    whitelist = {a for a in addrs if a.startswith("0x") and len(a) == 42}
    print(f"Token whitelist built: {len(whitelist)} ERC-20 tokens")
    return whitelist


# -----------------------------------
# CELL 3 — block resolver (same as yours)
# -----------------------------------
def resolve_blocks_for_dates(
    dates: List[str] | List[pd.Timestamp],
    block_to_date_csv: str,
) -> List[int]:
    if not dates:
        return []
    m = pd.read_csv(block_to_date_csv)
    block_col = next((c for c in ["block_number", "block"] if c in m.columns), None)
    if block_col is None:
        raise ValueError(f"Missing block column in {block_to_date_csv}. Got {list(m.columns)}")

    date_col = next((c for c in ["date", "bound_date", "block_time_utc"] if c in m.columns), None)
    if date_col is None:
        raise ValueError(f"Missing date column in {block_to_date_csv}. Got {list(m.columns)}")

    m[date_col] = pd.to_datetime(m[date_col], errors="coerce").dt.normalize()
    m = m.dropna(subset=[date_col, block_col])

    out_blocks: List[int] = []
    for d in dates:
        d_norm = pd.to_datetime(d, errors="coerce")
        if pd.isna(d_norm):
            print(f"[blocks] WARN could not parse date '{d}' (skip)")
            continue
        d_norm = d_norm.normalize()
        row = m.loc[m[date_col] == d_norm]
        if row.empty:
            print(f"[blocks] WARN no block for date {d_norm.date()} (skip)")
            continue
        out_blocks.append(int(row.iloc[0][block_col]))
    return out_blocks


# -------------------------------------------------
# CELL 4 — your snapshot list builder
# -------------------------------------------------
def build_dates_list() -> List[str]:
    dates: List[str] = []
    start = pd.Timestamp("2020-01-01")
    end   = pd.Timestamp("2025-12-01")
    cur = start
    while cur <= end:
        dates.append(cur.strftime("%Y-%m-%d"))
        cur = cur + pd.offsets.MonthBegin(1)

    # stable dedupe
    seen = set()
    merged = []
    for d in dates:
        if d not in seen:
            seen.add(d)
            merged.append(d)
    return merged


# ---------------------------------------------------------
# CELL 5 — parquet locator + schema detector (same as yours)
# ---------------------------------------------------------
def find_token_parquet_glob(data_root: str | pathlib.Path, token_addr: str) -> Optional[str]:
    root = pathlib.Path(data_root)
    addr = token_addr.lower()
    candidates = [
        str(root / f"token_address={addr}" / "*.parquet"),
        str(root / f"token_address={addr}.parquet"),
        str(root / addr / "*.parquet"),
        str(root / addr / "**" / "*.parquet"),
    ]
    for g in candidates:
        if glob.glob(g, recursive=True):
            return g
    return None

def detect_transfer_schema(con, parquet_glob: str):
    cols = list(con.execute(f"SELECT * FROM read_parquet('{parquet_glob}') LIMIT 0").df().columns)
    lower = {c.lower(): c for c in cols}

    def pick(cands):
        for c in cands:
            if c in lower:
                return lower[c]
        return None

    block_edge = pick(["block_number", "block", "blk_num"])
    from_col   = pick(["from_address", "from", "src"])
    to_col     = pick(["to_address", "to", "dst"])
    if block_edge and from_col and to_col:
        return {"mode": "edge", "block": block_edge, "from": from_col, "to": to_col}

    block_delta = pick(["block_number", "block", "blk_num"])
    addr_col    = pick(["address", "addr"])
    val_col     = pick(["value", "amount", "delta"])
    if block_delta and addr_col and val_col:
        return {"mode": "delta", "block": block_delta, "address": addr_col, "value": val_col}

    raise ValueError(f"Unsupported schema. Found columns: {cols}")



# -------------------------------------------------------
# CELL 7 — append-only tables (NO PK, NO upsert)
# -------------------------------------------------------
def reset_tables(con: duckdb.DuckDBPyConnection):
    con.execute("DROP TABLE IF EXISTS first_seen_raw;")
    con.execute("DROP TABLE IF EXISTS global_first_seen;")
    con.execute("""
        CREATE TABLE first_seen_raw (
            addr VARCHAR,
            first_seen_block BIGINT
        );
    """)

def append_first_seen_for_token(con, parquet_glob: str, schema: dict, max_block: int):
    max_block = int(max_block)
    con.execute("DROP TABLE IF EXISTS tmp_first_seen;")

    if schema["mode"] == "edge":
        b = schema["block"]; f = schema["from"]; t = schema["to"]
        con.execute(f"""
            CREATE TEMP TABLE tmp_first_seen AS
            SELECT lower(trim(addr)) AS addr, MIN(block_number) AS first_seen_block
            FROM (
                SELECT {b} AS block_number, {f} AS addr
                FROM read_parquet('{parquet_glob}')
                WHERE {f} IS NOT NULL AND {f} <> ''
                  AND {b} <= {max_block}

                UNION ALL

                SELECT {b} AS block_number, {t} AS addr
                FROM read_parquet('{parquet_glob}')
                WHERE {t} IS NOT NULL AND {t} <> ''
                  AND {b} <= {max_block}
            ) x
            GROUP BY 1;
        """)
    elif schema["mode"] == "delta":
        b = schema["block"]; a = schema["address"]
        con.execute(f"""
            CREATE TEMP TABLE tmp_first_seen AS
            SELECT lower(trim({a})) AS addr, MIN({b}) AS first_seen_block
            FROM read_parquet('{parquet_glob}')
            WHERE {a} IS NOT NULL AND {a} <> ''
              AND {b} <= {max_block}
            GROUP BY 1;
        """)
    else:
        raise ValueError(f"Unknown schema mode: {schema}")

    # append (no conflicts possible)
    con.execute("INSERT INTO first_seen_raw SELECT addr, first_seen_block FROM tmp_first_seen;")


def finalize_global_first_seen(con: duckdb.DuckDBPyConnection):
    con.execute("DROP TABLE IF EXISTS global_first_seen;")
    con.execute("""
        CREATE TABLE global_first_seen AS
        SELECT addr, MIN(first_seen_block) AS first_seen_block
        FROM first_seen_raw
        WHERE addr IS NOT NULL AND addr <> ''
        GROUP BY addr;
    """)
    con.execute("CREATE INDEX IF NOT EXISTS idx_global_first_seen_block ON global_first_seen(first_seen_block);")


In [2]:
# -----------------------------
# CELL 8 — paths & snapshots
# -----------------------------
import sys
sys.path.insert(0, "../../../shared")
from locations import Location

PORTFOLIO_RECONSTRUCTION_DATA = str(Location.ETH_TRANSFERS_DATA)
TOKEN_STATUS_CSV = str(Location.TOKEN_STATUS_PRIMARY_CSV)
BLOCK_TO_DATE_CSV = str(Location.BLOCK_TO_DATE_CSV)

CA_CREATION_FILE = str(Location.CA_CREATION_TSV)

OUTPUT_CSV = "new_output_accounts_eoa_vs_ca.csv"

date_list = build_dates_list()
snapshot_blocks = resolve_blocks_for_dates(date_list, BLOCK_TO_DATE_CSV)
pairs = [(d, b) for d, b in zip(date_list, snapshot_blocks) if b is not None]

target_blocks = [int(b) for _, b in pairs]
date_map = {int(b): str(d) for d, b in pairs}
MAX_BLOCK = max(target_blocks)

print("snapshots:", len(target_blocks), "MAX_BLOCK:", MAX_BLOCK)




snapshots: 72 MAX_BLOCK: 23914921


In [3]:
# ------------------------------------------
# CELL 9 — init db + load CA list (DuckDB native) + reset
# ------------------------------------------
whitelist = load_token_whitelist_from_status_csv(TOKEN_STATUS_CSV)
if not whitelist:
    raise RuntimeError("Whitelist empty.")

con = get_duck(DUCKDB_PATH, DUCK_THREADS, DUCK_MEM)

# clean run (recommended after crashes / changed MAX_BLOCK)
reset_tables(con)

# IMPORTANT: do NOT load 84M rows into pandas
con.execute("DROP TABLE IF EXISTS contract_creation;")

# Your file name says .tsv -> try delimiter = '\t' first
# If it's whitespace-separated instead, use delim=' ' and set sep.
con.execute(f"""
    CREATE TABLE contract_creation AS
    SELECT
        lower(trim(column0)) AS addr,
        CAST(column1 AS BIGINT) AS creation_block
    FROM read_csv(
        '{CA_CREATION_FILE}',
        delim='\\t',
        header=False,
        columns={{'column0':'VARCHAR', 'column1':'VARCHAR'}},
        ignore_errors=true
    )
    WHERE column0 IS NOT NULL
      AND column1 IS NOT NULL
      AND lower(trim(column0)) LIKE '0x%'
      AND length(trim(column0)) = 42
      AND try_cast(column1 AS BIGINT) IS NOT NULL
      AND try_cast(column1 AS BIGINT) <= {int(MAX_BLOCK)};
""")

# This index can be HUGE on 84M rows.
# I recommend keeping ONLY the block index (for <= t.block counting),
# and skipping addr index unless you truly need it.
con.execute("CREATE INDEX IF NOT EXISTS idx_contract_creation_block ON contract_creation(creation_block);")

print("contract_creation rows:",
      con.execute("SELECT COUNT(*) FROM contract_creation").fetchone()[0])


Token whitelist built: 4562 ERC-20 tokens
contract_creation rows: 86679789


In [4]:
# ---------------------------------------------------
# CELL 10 — scan tokens (append-only, no transactions)
# ---------------------------------------------------
processed = 0
skipped = 0
errors = 0
failed = []

for token in sorted(whitelist):
    g = find_token_parquet_glob(PORTFOLIO_RECONSTRUCTION_DATA, token)
    if not g:
        skipped += 1
        continue

    try:
        schema = detect_transfer_schema(con, g)
        append_first_seen_for_token(con, g, schema, max_block=MAX_BLOCK)

        processed += 1
        if processed % 25 == 0:
            n_raw = con.execute("SELECT COUNT(*) FROM first_seen_raw").fetchone()[0]
            print(f"[ok] tokens={processed} raw_rows={n_raw:,}")

    except Exception as e:
        errors += 1
        failed.append((token, str(e)))
        print(f"[err] token {token}: {e}")

print(f"Scan done. processed={processed}, skipped={skipped}, errors={errors}")
if failed:
    print("Example failed:", failed[0])





[ok] tokens=25 raw_rows=921,420
[ok] tokens=50 raw_rows=1,133,083
[ok] tokens=75 raw_rows=1,352,548
[ok] tokens=100 raw_rows=1,677,412
[ok] tokens=125 raw_rows=2,101,505
[ok] tokens=150 raw_rows=2,928,777
[ok] tokens=175 raw_rows=3,360,682
[ok] tokens=200 raw_rows=3,683,284
[ok] tokens=225 raw_rows=4,010,712
[ok] tokens=250 raw_rows=4,653,021
[ok] tokens=275 raw_rows=5,397,291
[ok] tokens=300 raw_rows=5,580,232
[ok] tokens=325 raw_rows=6,045,745
[ok] tokens=350 raw_rows=6,570,221
[ok] tokens=375 raw_rows=6,939,579
[ok] tokens=400 raw_rows=7,219,326
[ok] tokens=425 raw_rows=7,710,411
[ok] tokens=450 raw_rows=8,061,755
[ok] tokens=475 raw_rows=8,323,407
[ok] tokens=500 raw_rows=8,582,386
[ok] tokens=525 raw_rows=8,840,420
[ok] tokens=550 raw_rows=8,992,874
[ok] tokens=575 raw_rows=11,180,670
[ok] tokens=600 raw_rows=11,313,060
[ok] tokens=625 raw_rows=12,264,705
[ok] tokens=650 raw_rows=12,469,336
[ok] tokens=675 raw_rows=13,070,127
[ok] tokens=700 raw_rows=13,301,794
[ok] tokens=725 raw

In [5]:
# ---------------------------------------------------
# CELL 11 — finalize global_first_seen and aggregate (FASTER)
# ---------------------------------------------------
finalize_global_first_seen(con)

# Build targets temp table
vals = ", ".join(f"({int(b)})" for b in target_blocks)
con.execute("DROP TABLE IF EXISTS targets;")
con.execute("CREATE TEMP TABLE targets(block BIGINT);")
con.execute(f"INSERT INTO targets SELECT * FROM (VALUES {vals}) AS t(block);")

# --- Precompute CA accounts that are also observed in transfers ---
con.execute("DROP TABLE IF EXISTS ca_observed;")
con.execute("""
    CREATE TEMP TABLE ca_observed AS
    SELECT
        g.addr,
        GREATEST(g.first_seen_block, cc.creation_block) AS ca_effective_block
    FROM global_first_seen g
    JOIN contract_creation cc
      ON cc.addr = g.addr;
""")

# Index helps for counting <= t.block
con.execute("CREATE INDEX IF NOT EXISTS idx_ca_observed_eff ON ca_observed(ca_effective_block);")

df_counts = con.execute("""
SELECT
  t.block AS block,

  -- total observed addresses by that snapshot
  (
    SELECT COUNT(*)
    FROM global_first_seen g
    WHERE g.first_seen_block <= t.block
  ) AS total_accounts,

  -- CA observed by snapshot (both created and seen)
  (
    SELECT COUNT(*)
    FROM ca_observed c
    WHERE c.ca_effective_block <= t.block
  ) AS ca_accounts

FROM targets t
ORDER BY t.block;
""").df()

df_counts["eoa_accounts"] = df_counts["total_accounts"] - df_counts["ca_accounts"]

df_counts["date"] = df_counts["block"].map(date_map)
df_counts = df_counts[["block", "date", "eoa_accounts", "ca_accounts"]]
df_counts.to_csv(OUTPUT_CSV, index=False)

print("Wrote:", OUTPUT_CSV)
df_counts.head(10)


Wrote: new_output_accounts_eoa_vs_ca.csv


,block,date,eoa_accounts,ca_accounts
0,9193266,2020-01-01,9694262,498758
1,9393154,2020-02-01,10092848,559130
2,9581792,2020-03-01,10689553,638957
3,9782602,2020-04-01,11456511,769438
4,9976964,2020-05-01,12676503,974964
5,10176690,2020-06-01,13934252,1285486
6,10370274,2020-07-01,15140553,1507797
7,10570485,2020-08-01,16417543,1616078
8,10771925,2020-09-01,17905532,1686019
9,10966874,2020-10-01,19177652,1744197
